# Bachelor Thesis

© 2026 Yvan Richard   
University of St. Gallen, Spring Term 2026

## Exploring Hypothesis 1

---
## Foreword

In this notebook, my goal is to explore this hypothesis:

$$
\mathbb{E}[\Delta\textit{users\_close}_{i, t + 1} \mid \text{high}(ASVI_{i, t})] \ge 0
$$

where $ASVI_{i, t}$ translates abnormal volume in Google Trends. In other words, the purpose of this hypothesis is to explore whether lagged attention spikes in Google Trends can predict an increase in the number of Robinhood users holding a stock $i$ at time $t+1$. Additionally, I will also focus on the following question: "Among competing attention proxies, which ones contain incremental information about next-day retail demand on Robinhood?"


## 1. Libraries & Data

I first load the relevant libraries and data.

In [ ]:
# libs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from linearmodels import OLS

In [5]:
# data
main = pd.read_csv("../../../data/processed/main_with_svi.csv")

/var/folders/7v/_v_y1jpx0rl056gg5rkjsw4r0000gn/T/ipykernel_6194/1285260102.py:2: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  main = pd.read_csv("../../../data/processed/main_with_svi.csv")


Once I loaded the data, I parse the date accordingly.

In [7]:
# date format
main["date"] = pd.to_datetime(main["date"])

## 2. Exploratory Data Analysis

In this section, I conduct a brief EDA.

In [9]:
# the columns
main.columns

Index(['date', 'users_close', 'users_last', 'ticker', 'permno', 'ret', 'prc',
       'vol', 'shrout', 'exchcd', 'symro', 'symsu', 'buy_num_trades_LR',
       'sell_num_trades_LR', 'total_trade_LR', 'buy_vol_LR', 'sell_vol_LR',
       'total_vol_LR', 'close_price', 'open_price', 'close_vol', 'open_vol',
       'total_vol_m', 'intra_ret', 'buy_num_trades_tick',
       'sell_num_trades_tick', 'buy_vol_tick', 'sell_vol_tick',
       'total_trade_tick', 'buy_num_trades_wrds', 'sell_num_trades_wrds',
       'buy_vol_wrds', 'sell_vol_wrds', 'total_trade_wrds', 'bs_ratio_num',
       'bs_ratio_vol', 'buy_num_trades_retail', 'sell_num_trades_retail',
       'buy_vol_retail', 'sell_vol_retail', 'total_trade_retail',
       'total_vol_retail', 'bs_ratio_retail_vol', 'bs_ratio_retail_num',
       'intra_volatility', 'buy_num_trades_inst50k', 'sell_num_trades_inst50k',
       'buy_vol_inst50k', 'sell_vol_inst50k', 'total_trade_inst50k',
       'total_vol_inst50k', 'bs_ratio_inst50k_vol', 'bs_ratio_

In [10]:
# rename TSSVI to svi
main.rename(columns={"TSSVI": "svi"}, inplace=True)

In [12]:
# number of non-nan rows with svi and number of unique tickers in those rows
print(f"Number of non-nan rows with svi: {main['svi'].notna().sum()}")
print(f"Number of unique tickers in those rows: {main[main['date'].notna() & main['svi'].notna()]['ticker'].nunique()}")

Number of non-nan rows with svi: 871692
Number of unique tickers in those rows: 1698


## 3. Regressions

### 3.1. Basline Regression (Abnormal Volume and Returns)

The first regression is specifed as follows.

**Dependent Variable**

$$
\Delta \textit{users\_close}_{i, t} = {\textit{users\_close}_{i, t} - \textit{users\_close}_{i, t -1}}
$$

**Independent Variables**

1. abnormal returns

$$
\textit{ab\_ret}_{i, t - 1} = \ln(\frac{ret_{i, t - 1}}{\frac{1}{20}\sum_{k = 2}^{21} ret_{i, t - k}})
$$

2. abnormal volume

$$
\textit{ab\_vol}_{i, t} = \ln(\frac{vol_{i, t - 1}}{\frac{1}{20}\sum_{k = 2}^{21} vol_{i, t - k}})
$$

In [40]:
# select only relevant variables
df_reg_1 = main[['date', 'ticker', 'users_close', 'ret', 'vol', 'svi', 'num_news']].copy()

# sort by ticker and date
df_reg_1.sort_values(by=['ticker', 'date'], inplace=True)

# drop rows with missing raw inputs
df_reg_1.dropna(subset=['users_close', 'ret', 'vol'], inplace=True)

# userchg
df_reg_1['userchg'] = df_reg_1.groupby('ticker')['users_close'].diff()

# userchg(t - 1)
df_reg_1['userchg_lag1'] = df_reg_1.groupby('ticker')['userchg'].shift(1)

# ln userchg
df_reg_1['ln_userchg'] = np.log(1 + np.abs(df_reg_1['userchg']))

# ab_ret_lag1
def make_ab_ret(s):
    lags = pd.concat([s.shift(k) for k in range(2, 22)], axis=1)
    mu = lags.mean(axis=1)
    sigma = lags.std(axis=1)
    z = (s.shift(1) - mu) / sigma
    z[sigma == 0] = np.nan
    return z

df_reg_1["ab_ret_lag1"] = df_reg_1.groupby("ticker")["ret"].transform(make_ab_ret)

# ret(t - 1)
df_reg_1['ret_lag1'] = df_reg_1.groupby('ticker')['ret'].shift(1)

# ab_vol_lag1
def make_ab_vol(s):
    lags = pd.concat([s.shift(k) for k in range(2, 22)], axis=1)
    mu = lags.mean(axis=1)
    ratio = s.shift(1) / mu
    ratio[mu == 0] = np.nan
    return np.log1p(np.abs(ratio))

df_reg_1["ab_vol_lag1"] = df_reg_1.groupby("ticker")["vol"].transform(make_ab_vol)

# abnormal svi: ln(svi(t - 1) / mean(svi(t - 2), ..., svi(t - 21)))
def make_ab_svi(s):
    lags = pd.concat([s.shift(k) for k in range(2, 22)], axis=1)
    mu = lags.mean(axis=1)
    ratio = s.shift(1) / mu
    ratio[mu == 0] = np.nan
    return np.log1p(np.abs(ratio))

df_reg_1["ab_svi_lag1"] = df_reg_1.groupby("ticker")["svi"].transform(make_ab_svi)

# abnormal news: ln(num_news(t - 1) / mean(num_news(t - 2), ..., num_news(t - 21)))
def make_ab_news(s):
    lags = pd.concat([s.shift(k) for k in range(2, 22)], axis=1)
    mu = lags.mean(axis=1)
    ratio = s.shift(1) / mu
    ratio[mu == 0] = np.nan
    return np.log1p(np.abs(ratio))

df_reg_1["ab_news_lag1"] = df_reg_1.groupby("ticker")["num_news"].transform(make_ab_news)

# optional cleanup
df_reg_1.replace([np.inf, -np.inf], np.nan, inplace=True)

In [41]:
reg_data = df_reg_1.dropna().copy()

# make sure date is datetime
reg_data['date'] = pd.to_datetime(reg_data['date'])

# convert clusters to integer codes
reg_data['ticker_clust'] = reg_data['ticker'].astype('category').cat.codes
reg_data['date_clust'] = reg_data['date'].astype('category').cat.codes

X1 = sm.add_constant(reg_data[['ret_lag1', 'ab_vol_lag1', 'userchg_lag1']])
y1 = reg_data['userchg']

results1 = OLS(y1, X1).fit(
    cov_type='clustered',
    clusters=reg_data[['ticker_clust', 'date_clust']]
)

print(results1.summary)

                            OLS Estimation Summary                            
Dep. Variable:                userchg   R-squared:                      0.2240
Estimator:                        OLS   Adj. R-squared:                 0.2240
No. Observations:              359732   F-statistic:                    164.00
Date:                Fri, Mar 20 2026   P-value (F-stat)                0.0000
Time:                        16:38:08   Distribution:                  chi2(3)
Cov. Estimator:             clustered                                         
                                                                              
                              Parameter Estimates                               
              Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------
const            0.4827     1.6887     0.2858     0.7750     -2.8271      3.7924
ret_lag1         186.79     93.693     1.993

In [42]:
X2 = sm.add_constant(reg_data[['ret_lag1', 'ab_vol_lag1', 'userchg_lag1', 'ab_svi_lag1']])
y2 = reg_data['userchg']

results2 = OLS(y2, X2).fit(
    cov_type='clustered',
    clusters=reg_data[['ticker_clust', 'date_clust']]
)

print(results2.summary)

                            OLS Estimation Summary                            
Dep. Variable:                userchg   R-squared:                      0.2241
Estimator:                        OLS   Adj. R-squared:                 0.2240
No. Observations:              359732   F-statistic:                    191.85
Date:                Fri, Mar 20 2026   P-value (F-stat)                0.0000
Time:                        16:39:22   Distribution:                  chi2(4)
Cov. Estimator:             clustered                                         
                                                                              
                              Parameter Estimates                               
              Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------
const            0.0376     1.6214     0.0232     0.9815     -3.1404      3.2155
ret_lag1         186.65     93.700     1.992

In [43]:
X3 = sm.add_constant(reg_data[['ret_lag1', 'ab_vol_lag1', 'userchg_lag1', 'ab_svi_lag1', 'ab_news_lag1']])
y3 = reg_data['userchg']

results3 = OLS(y3, X3).fit(
    cov_type='clustered',
    clusters=reg_data[['ticker_clust', 'date_clust']]
)

print(results3.summary)

                            OLS Estimation Summary                            
Dep. Variable:                userchg   R-squared:                      0.2241
Estimator:                        OLS   Adj. R-squared:                 0.2240
No. Observations:              359732   F-statistic:                    328.61
Date:                Fri, Mar 20 2026   P-value (F-stat)                0.0000
Time:                        16:40:46   Distribution:                  chi2(5)
Cov. Estimator:             clustered                                         
                                                                              
                              Parameter Estimates                               
              Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------
const           -0.0424     1.5057    -0.0282     0.9775     -2.9935      2.9086
ret_lag1         186.67     93.633     1.993